In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

df_sales = pd.read_csv('/content/games_sales.csv')

df_sales.isnull().sum()

,0
Name,2
Platform,0
Year_of_Release,269
Genre,2
Publisher,54
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0
Global_Sales,0


In [2]:
# จัดการข้อมูลประเภทตัวเลข (Numerical Cleaning)
df_sales['User_Score'] = pd.to_numeric(df_sales['User_Score'], errors='coerce')
df_sales['Year_of_Release'] = pd.to_numeric(df_sales['Year_of_Release'], errors='coerce')

# เติมค่าว่างด้วยค่ากลาง (Median) แยกตามแนวเกม (Genre) เพื่อความแม่นยำสูง
df_sales['Critic_Score'] = df_sales['Critic_Score'].fillna(df_sales.groupby('Genre')['Critic_Score'].transform('median'))
df_sales['User_Score'] = df_sales['User_Score'].fillna(df_sales.groupby('Genre')['User_Score'].transform('median'))

# เติมค่าว่างด้วย Mode
df_sales['Year_of_Release'] = df_sales['Year_of_Release'].fillna(df_sales['Year_of_Release'].mode()[0])

# เก็บตกค่าในช่องที่ใช้ ค่ากลาง (Median) แยกตามแนวเกม (Genre) ไม่ได้ ให้ใช้ Med ของคอลัมน์นั้นแทน
df_sales['Critic_Score'] = df_sales['Critic_Score'].fillna(df_sales['Critic_Score'].median())
df_sales['User_Score'] = df_sales['User_Score'].fillna(df_sales['User_Score'].median())

# จัดการข้อมูลประเภทข้อความ (Categorical Cleaning) เติมช่องว่างด้วย 'Unknown' เพื่อสร้างกลุ่มใหม่
cols_to_fix = ['Publisher', 'Developer', 'Rating', 'Genre', 'Platform','Name']
for col in cols_to_fix:
    df_sales[col] = df_sales[col].fillna('Unknown')

# สร้าง Target สำหรับ Classification
def label_success(sales):
    if sales > 10: return 3    # AAA Hit
    if sales > 1:  return 2    # Successful
    if sales > 0.1: return 1   # Mid-range
    return 0                   # Low-sales

df_sales['success_level'] = df_sales['Global_Sales'].apply(label_success)

# 5. Feature Engineering & Encoding
# ใช้ One-Hot Encoding สำหรับ Genre และ Platform เพื่อให้ Neural Network เรียนรู้ได้ดีขึ้น
X = pd.get_dummies(df_sales[['Genre', 'Platform', 'Critic_Score', 'User_Score', 'Year_of_Release']],
                   columns=['Genre', 'Platform'])
y = df_sales['success_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

df_sales.isnull().sum()

,0
Name,0
Platform,0
Year_of_Release,0
Genre,0
Publisher,0
NA_Sales,0
EU_Sales,0
JP_Sales,0
Other_Sales,0
Global_Sales,0


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model_nn = Sequential([
    Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(4, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model_nn.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model_nn.fit(
    X_train_scaled, y_train,
    epochs=60,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/60
168/168 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.5507 - loss: 0.9441 - val_accuracy: 0.5720 - val_loss: 0.9004
Epoch 2/60
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5829 - loss: 0.8732 - val_accuracy: 0.5798 - val_loss: 0.8695
Epoch 3/60
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5935 - loss: 0.8514 - val_accuracy: 0.5880 - val_loss: 0.8598
Epoch 4/60
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5936 - loss: 0.8401 - val_accuracy: 0.5824 - val_loss: 0.8603
Epoch 5/60
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5995 - loss: 0.8299 - val_accuracy: 0.5895 - val_loss: 0.8491
Epoch 6/60
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6033 - loss: 0.8259 - val_accuracy: 0.5847 - val_loss: 0.8461
Epoch 7/60
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6054 - loss: 0.8194 - val_accuracy: 0.5955 - val_loss: 0.8397
Epoch 8/60
168/168 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6111 - loss: 0.8134 - val_accuracy: 0.

In [6]:
loss, accuracy = model_nn.evaluate(X_test_scaled, y_test)
print(f"Neural Network Accuracy: {accuracy*100:.2f}%")

105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6026 - loss: 0.8478
Neural Network Accuracy: 60.26%


In [7]:
import joblib
from google.colab import files

model_columns_nn = X.columns.tolist()
joblib.dump(model_columns_nn, 'model_columns_nn.pkl')

joblib.dump(scaler, 'scaler_nn.pkl')
model_nn.save('game_nn_model.h5')

files.download('game_nn_model.h5')
files.download('scaler_nn.pkl')
files.download('model_columns_nn.pkl')

df_sales.to_csv('cleaned_games_sales.csv', index=False)
files.download('cleaned_games_sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>